In [ ]:
import csv
import os
import subprocess
from string import punctuation

import pandas as pd

import spacy
# import spacy_transformers
# import spacy_curated_transformers
# from spacy_curated_transformers
from wordfreq import zipf_frequency

In [ ]:
# Cell 2
def count_csv_elements_in_file(filepath):

    total_elements = 0

    try:
        with open(filepath, "r", encoding="utf-8") as file:
            csv_reader = csv.reader(file)
            for row in csv_reader:
                total_elements += len(row)
    except (UnicodeDecodeError, IOError):
        # Skip files that can't be read as UTF-8 text
        pass

    return total_elements

language = []
total_words = []

for path, subdirs, files in os.walk("raw-word-lists"):
    for name in files:
        # Only process .txt files; skip .DS_Store and README.md
        if not name.endswith(".txt"):
            continue
        filename = (os.path.join(path, name))
        language += [name.split(".")[0]]
        total_words += [count_csv_elements_in_file(filename)]

In [ ]:
# Cell 3
pd.DataFrame({
    "language": language,
    "total_words": total_words
})

In [ ]:
# Cell 4
spacy_models = {
    "Catalan": "ca_core_news_trf",
    "Croatian": "hr_core_news_lg",
    "Danish": "da_core_news_trf",
    "Dutch": "nl_core_news_lg",
    "English": "en_core_web_trf",
    "Finnish": "fi_core_news_lg",
    "French": "fr_dep_news_trf",
    "German": "de_dep_news_trf",
    "Greek": "el_core_news_lg",
    "Italian": "it_core_news_lg",
    "Lithuanian": "lt_core_news_lg",
    "Romanian": "ro_core_news_lg",
    "Russian": "ru_core_news_lg",
    "Slovenian": "sl_core_news_trf",
    "Spanish": "es_dep_news_trf",
    "Swedish": "sv_core_news_lg",
    "Ukrainian": "uk_core_news_trf"
}

In [ ]:
# Cell 5
for model in spacy_models.values():
    result = subprocess.run(['python', '-m', 'spacy', 'download', f'{model}'],
                       capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)

In [ ]:
# Cell 6
try:
    os.mkdir('data')
except FileExistsError:
    print('File or directory data already exists')

In [ ]:
# Cell 7
for language in spacy_models.keys():
    try:
        os.mkdir(f"data/{language}")
        print(f"Directory {language} created")
    except FileExistsError:
        print(f"Directory {language} already exists")

In [ ]:
# Cell 8
def load_and_clean_word_list(language: str) -> pd.DataFrame:
    with open(f"raw-word-lists/{language}/{language}.txt", "r", encoding="utf-8") as f:
        word_list = f.read().split(",")

    word_df = pd.DataFrame({
        "word": word_list
    })

    word_df["word"] = word_df["word"].str.strip(punctuation)
    word_df = word_df[word_df["word"].str.len() > 0].reset_index(drop=True)

    return word_df

In [ ]:
# Cell 9
load_and_clean_word_list("English")

In [ ]:
nlp = spacy.load(spacy_models["English"], disable=["parser", "ner", "textcat"])

In [ ]:
# Cell 11
from math import ceil
from tqdm.auto import tqdm

def add_lemma(df: pd.DataFrame,
              nlp,
              batch_size: int = 1000) -> pd.DataFrame:
    total_batches = ceil(len(df) / batch_size)
    docs = tqdm(
        nlp.pipe(df["word"].tolist(), batch_size=batch_size),
        total=total_batches,
        desc="Lemmatizing",
        leave=False
    )
    lemmas = [doc[0].lemma_ if len(doc) > 0 else "" for doc in docs]
    df["lemma"] = pd.DataFrame(lemmas, index=df.index)
    return df

In [ ]:
# Cell 12
add_lemma(
    load_and_clean_word_list("English")[:100], nlp
)

In [ ]:
# Cell 13
def add_word_frequencies(df: pd.DataFrame,
                         language: str) -> pd.DataFrame:
    language_group = spacy_models[language].split("_")[0]
    df["zipf_freq_lemma"] = [zipf_frequency(w, language_group) for w in df["lemma"]]
    return df

In [ ]:
# Cell 14
add_word_frequencies(
    add_lemma(
        load_and_clean_word_list("English")[:100], nlp
), "English")

In [ ]:
# Cell 15
def clean_up_and_export(df: pd.DataFrame, language: str) -> None:
    df = (
        df.loc[df.groupby("lemma", sort=False)["zipf_freq_lemma"].idxmax()]
        .reset_index(drop=True)
    )

    df = df[(df["zipf_freq_lemma"] > 0)]

    df.loc[:, "word_difficulty"] = pd.cut(
        df["zipf_freq_lemma"],
        bins = [-float("inf"), 2.0, 4.0, float("inf")],
        labels = ["advanced", "intermediate", "beginner"],
        include_lowest = True,
        right = True
    )

    df = df.drop(columns=["word", "zipf_freq_lemma"])
    df = df.rename(columns = {
        "lemma": "word"
    })

    df.to_json(f"data/{language}/word-list-cleaned.json", orient="index")

In [ ]:
# Cell 16
clean_up_and_export(
    add_word_frequencies(
        add_lemma(
            load_and_clean_word_list("English")[:100], nlp),
        "English"),
    "English")

In [ ]:
# Cell 17
def create_clean_word_list(language: str) -> None:
    nlp = spacy.load(spacy_models[language], disable=["parser", "ner", "textcat"])

    print("Load in dataset")
    lang_df = load_and_clean_word_list(language)

    print("Lemmatise words")
    lang_df = add_lemma(lang_df, nlp)

    print("Add the word frequencies")
    lang_df = add_word_frequencies(lang_df, language)

    print("Do the final clean ups and export to file")
    clean_up_and_export(lang_df, language)

    return None

In [ ]:
# Cell 18
create_clean_word_list("Spanish")

In [ ]:
# Cell 19
import json

language_raw = []
total_words_raw = []

for path, subdirs, files in os.walk("raw-word-lists"):
    for name in files:
        if not name.endswith(".txt"):
            continue

        filename = (os.path.join(path, name))
        language_raw += [name.split(".")[0]]
        total_words_raw += [count_csv_elements_in_file(filename)]

raw_data = pd.DataFrame({
    "language": language_raw,
    "type": ["Raw"] * len(language_raw),
    "total_words_raw": total_words_raw,
})

In [ ]:
# Cell 20

language_clean = []
total_words_clean = []

for path, subdirs, files in os.walk("data"):
    for name in files:

        if not name.endswith(".txt"):
            continue

        filename = (os.path.join(path, name))
        language_clean += [path.split("/")[1]]
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
            total_words_clean += [len(data.keys())]

clean_data = pd.DataFrame({
    "language": language_raw,
    "type": ["Raw"] * len(language_raw),
    "total_words_raw": total_words_raw,
})

pd.concat([raw_data, clean_data])

In [81]:
# Cell 21

from tqdm.auto import tqdm

raw_word_lists_dir = "raw-word-lists"
languages = sorted([
    d for d in os.listdir(raw_word_lists_dir)
    if os.path.isdir(os.path.join(raw_word_lists_dir, d)) and d in spacy_models
])

for language in tqdm(languages, desc="Processing languages"):
    output_file = f"data/{language}/word-list-cleaned.json"

    # Skip if already processed unless manually deleted
    if os.path.exists(output_file):
        print(f"Skipping {language} (already processed: {output_file})...")
        continue

    print(f"Processing {language}...")
    create_clean_word_list(language)

Processing languages:   0%|          | 0/17 [00:00<?, ?it/s]

Skipping Catalan (already processed: data/Catalan/word-list-cleaned.json)...
Skipping Croatian (already processed: data/Croatian/word-list-cleaned.json)...
Skipping Danish (already processed: data/Danish/word-list-cleaned.json)...
Skipping Dutch (already processed: data/Dutch/word-list-cleaned.json)...
Skipping English (already processed: data/English/word-list-cleaned.json)...
Skipping Finnish (already processed: data/Finnish/word-list-cleaned.json)...
Skipping French (already processed: data/French/word-list-cleaned.json)...
Skipping German (already processed: data/German/word-list-cleaned.json)...
Skipping Greek (already processed: data/Greek/word-list-cleaned.json)...
Skipping Italian (already processed: data/Italian/word-list-cleaned.json)...
Processing Lithuanian...
Load in dataset
Lemmatise words


Lemmatizing:   0%|          | 0/83 [00:00<?, ?it/s]

Add the word frequencies
Do the final clean ups and export to file
Processing Romanian...
Load in dataset
Lemmatise words


Lemmatizing:   0%|          | 0/273 [00:00<?, ?it/s]

Add the word frequencies
Do the final clean ups and export to file
Processing Russian...
Load in dataset
Lemmatise words


... (4 outputs left out)

Lemmatizing:   0%|          | 0/404 [00:00<?, ?it/s]

Add the word frequencies
Do the final clean ups and export to file
Processing Ukrainian...
Load in dataset
Lemmatise words


Lemmatizing:   0%|          | 0/127 [00:00<?, ?it/s]

Add the word frequencies
Do the final clean ups and export to file
